<a href="https://colab.research.google.com/github/mbudisic/AIE6/blob/main/testing_rewards.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Debugging reward functions

We suggest improvements to the XML pattern matching format in AIE6 Session notebook 10.


## TLDR

Using the following pattern in `strict_format_reward_func`

```python
pattern = r"^<reasoning>\n(?:.*\n)*?</reasoning>\n<answer>\n(?:.*\n)*?</answer>\n?$"
```

and in `soft_format_reward_func`

```python
pattern = r"<reasoning>\s*[\s\S]*?\s*</reasoning>\s*<answer>\s*[\s\S]*?\s*</answer>"
```

results in a more accurate fine-tuned model that respects the XML format (`<reasoning>...</reasoning><answer>...</answer>`) of the response more closely.

For explanation of the regexes, see [soft on regex101](https://regex101.com/r/9ZRMK9/1) and [strict on regex101](https://regex101.com/r/RhojSp/1)).



(Note: the strict pattern has been updated and now it displays motion when training as well.)



## Explanation

During the AIE6 Session 10, our discussion group noticed that the
metrics `strict_format_reward_func` and `soft_format_reward_func` flatlined
at 0 during the training (meaning they were always reported that the outputs
do not conform with the `<reasoning>...</reasoning><answer>...</answer>` format).

For example, the following answer was generated. It was mathematically correct, but the XML pattern is not right - the tags are not closed and the answer is outside `<answer>` tags:

```
<reasoning>
To calculate pi (π), we can use the infinite series expansion, which is given by:

π = 1 + 1/3 + 1/5 + 1/7 + ...

This series is known as the Leibniz formula for π. To calculate pi to a certain number of decimal places, we can add up the terms of this series until we reach the desired accuracy.

For example, to calculate pi to 3 decimal places, we can add the first 10 terms of the series:

π ≈ 1 + 1/3 + 1/5 + 1/7 + 1/9 + 1/11 + 1/13 + 1/15 + 1/17 + 1/19
= 1 + 0.3333 + 0.2 + 0.1429 + 0.1111 + 0.0909 + 0.0769 + 0.0667 + 0.0588 + 0.0526
= 3.1415926

So, π is approximately 3.1415926 to 3 decimal places.

</answer>
3.1415926
```


While it is possible that the model simply didn't evolve far enough to
achieve any progress in this direction, another possibility is that the
reward function has some error which makes it always report a `0` (or some
other constant).
A reward function that is always equal to a constant is not useful,
as it doesn't reflect changes in the model and therefore does not provide a
signal to the optimization loop that the model is/isn't improving.

The following code can be used to debug the reward functions

In [1]:
import re

# Change from True -> False to substitute Wiz's Regex
ORIGINAL = True

def strict_format_reward_func(text, **kwargs) -> list[float]:
    """
    This function rewards responses that exactly follow a strict XML format.
    Steps:
    - It uses a regular expression pattern that enforces the precise structure:
      The response must start with <reasoning> on its own line, followed by some text,
      then </reasoning> on its own line, then <answer> on its own line, some text,
      and finally </answer> on its own line, with no extra content before or after.
    - It returns a reward of 0.5 if the response exactly matches this pattern, otherwise 0.0.
    """
    if (ORIGINAL):
      pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    else:
      # OLDpattern = r"^<reasoning>[\r\n|\r|\n]\s*[\s\S]*?[\r\n|\r|\n]</reasoning>[\r\n|\r|\n]<answer>[\r\n|\r|\n]\s*[\s\S]*?[\r\n|\r|\n]</answer>[\r\n|\r|\n]$"
      pattern = r"^<reasoning>\n(?:.*\n)*?</reasoning>\n<answer>\n(?:.*\n)*?</answer>\n?$"

    print(pattern)
    responses = text
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(text, **kwargs) -> list[float]:
    """
    This function rewards responses that generally follow the expected XML format.
    Steps:
    - It uses a more relaxed regular expression pattern that checks for the presence of <reasoning> and </reasoning>
      followed by <answer> and </answer> somewhere in the text.
    - It returns a reward of 0.5 if the pattern is found, otherwise 0.0.
    """
    if (ORIGINAL):
      pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    else:
      pattern = r"<reasoning>\s*[\s\S]*?\s*</reasoning>\s*<answer>\s*[\s\S]*?\s*</answer>"
    print(pattern)
    responses = text
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]




The code has been lifted and slightly modified from the Data Preparation section
of the class notebook to help with testing. First, we focus on the case
`ORIGINAL=True` which matches the code in the original notebook.

`soft_format_reward_func` looks for the pattern in the responses
that match

```xml
<reasoning>Hello</reasoning><answer>World</answer>
```

while the `strict_format_reward_function` is more rigid and wants everything lined up at the beginning of each line.

```xml
<reasoning>
Hello
</reasoning>
<answer>
World
</answer>

```

We can define a data set and test the function.

In [2]:
def test():
  print("ORIGINAL" if ORIGINAL else "MODIFIED")

  test_data = [ """<reasoning>
Hello
</reasoning>
<answer>
World
</answer>
""", # end \n is significant!
  """<reasoning>Hello</reasoning><answer>World</answer>""",
  """<reasoning>Hello</messedup>World</answer>""",
  ]

  [(print(f"*** {i} ***"),print(l),print("****")) for i,l in enumerate(test_data)]

  print( f"Strict: {strict_format_reward_func(test_data)}" )

  print( f"Soft: {soft_format_reward_func(test_data)}" )


If we set `ORIGINAL=True` and test, we see that the strict function never responds positively, while the soft function does, but only for the second case.

In [3]:
ORIGINAL=True
test()

ORIGINAL
*** 0 ***
<reasoning>
Hello
</reasoning>
<answer>
World
</answer>

****
*** 1 ***
<reasoning>Hello</reasoning><answer>World</answer>
****
*** 2 ***
<reasoning>Hello</messedup>World</answer>
****
^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$
Strict: [0.5, 0.0, 0.0]
<reasoning>.*?</reasoning>\s*<answer>.*?</answer>
Soft: [0.0, 0.5, 0.0]


In particular, the first pattern that should be scored as valid by `soft` is given a score of 0 (even though `soft` is supposed to be less strict than `strict`).

Wiz suggested that there may be a problem with the `regex` pattern, which is used to look for the XML tags. In particular, he suggested to replace the pattern in the `soft` function

```
<reasoning>.*?</reasoning>\s*<answer>.*?</answer>
```

with

```
<reasoning>\s*[\s\S]*?\s*</reasoning>\s*<answer>\s*[\s\S]*?\s*</answer>
```

(see [regex101](https://regex101.com/r/9ZRMK9/1) for explanation)

In short, this change moves from capturing _all_ characters inside XML tags to capturing only whitespace and non-special characters. The difference is technical and may relate to how Colab records newlines. 

The following change is made to the pattern in `strict`.

```python
pattern = r"^<reasoning>\n(?:.*\n)*?</reasoning>\n<answer>\n(?:.*\n)*?</answer>\n?$"
```

([see regex101 for explanation](https://regex101.com/r/RhojSp/1)).
Changing to the new pattern by setting `ORIGINAL=False` we see that

In [4]:
ORIGINAL=False
test()

MODIFIED
*** 0 ***
<reasoning>
Hello
</reasoning>
<answer>
World
</answer>

****
*** 1 ***
<reasoning>Hello</reasoning><answer>World</answer>
****
*** 2 ***
<reasoning>Hello</messedup>World</answer>
****
^<reasoning>\n(?:.*\n)*?</reasoning>\n<answer>\n(?:.*\n)*?</answer>\n?$
Strict: [0.5, 0.0, 0.0]
<reasoning>\s*[\s\S]*?\s*</reasoning>\s*<answer>\s*[\s\S]*?\s*</answer>
Soft: [0.5, 0.5, 0.0]


## Summary

Running the training with the new pattern resulted in the following graph of the soft reward function:


![rewards graphs](./rewards_graphs.png)

An example of the output is:
    
```
<reasoning>
To calculate pi (π), I'll use the Bailey-Borwein-Plouffe (BBP) formula, a spigot algorithm for computing the nth binary digit of pi. This algorithm is an extension of the Gauss-Legendre algorithm and is known for its high precision.

The BBP formula is as follows:

π = Σ (1/(16^k)) \* ((4/(8k+1)) - (2/(8k+4)) - (1/(8k+5)) - (1/(8k+6)))

where k = 0, 1, 2, 3, ...

I'll use this formula to calculate the first 10 digits of pi.

</reasoning>
<answer>
3.1415926535897932384626433832795028841971693993751058209749445923078164062862089986280348253421170679
</answer>
```

In particular, notice that the `xml` tags are formatted as expected.

In [11]:
import pandas as pd 
import wandb
api = wandb.Api()

# Project is specified by <entity/project-name>
runs = api.runs("budisicm-virginia-commonwealth-university/huggingface")

summary_list, config_list, name_list = [], [], []
for run in runs: 
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files 
    summary_list.append(run.summary._json_dict)

    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append(
        {k: v for k,v in run.config.items()
          if not k.startswith('_')})

    # .name is the human-readable name of the run.
    name_list.append(run.name)

runs_df = pd.DataFrame({
    "summary": summary_list,
    "config": config_list,
    "name": name_list
    })


